# ESCM — baseline network

Reconstruct the original network from recorded YARP camera events. Keep the legacy neural dynamics until behavior is validated.

In [ ]:
from pathlib import Path
import numpy as np
from brian2 import *
from escm.event_input import load_events

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
# Original Human3.6m Directions sample ("Directions_HPE" in ESCM_W.py).
recording = load_events(
    root / "InputData/h36m_sample/cam2_S1_Directions/ch0dvs",
    channel="left", width=640, height=480,
    timestamp_clock_ns=1000,  # Verify the timestamp units and channel
)
events = recording["events"]

## Camera spikes

Match the legacy simulation parameters, including `timeScale = 0.1` in `BrianHF.event_to_spike`. The loader itself preserves raw timestamps; scaling happens only when building the simulation input. The default is the **full recording**; set `duration = 0.1` for a quick 100 ms test. Note that legacy event preprocessing may differ from this adapter (particularly timestamp alignment and duplicate handling).

In [ ]:
start_scope()
defaultclock.dt = 0.5 * ms
width, height = recording["width"], recording["height"]
N = width * height
tau, vt, vr = 0.2 * ms, 0.1, 0.0
beta, Wi, Wk, k = 0.5, 6.0, -3.0, 8
time_scale = 0.1  # Legacy ESCM_W.py uses 1/10.0
duration = None   # Full scaled recording; use 0.1 for a short test

ts = np.asarray(events["ts"])
if not len(ts):
    raise ValueError("Empty event stream")
t_scaled = (ts - ts[0]) * time_scale
use = np.ones(len(ts), dtype=bool) if duration is None else t_scaled < duration
pixel = np.asarray(events["y"])[use] * width + np.asarray(events["x"])[use]
ticks = np.floor(t_scaled[use] / float(defaultclock.dt / second)).astype(np.int64)
pairs = np.unique(np.column_stack((ticks, pixel)), axis=0)
input_duration = float((ticks.max() + 1) * defaultclock.dt / second) if len(ticks) else 0.0
if duration is None:
    duration = input_duration
camera = SpikeGeneratorGroup(N, pairs[:, 1], pairs[:, 0] * defaultclock.dt, sorted=True)

## Network

Direct full-field connectivity avoids the legacy chunk-boundary omission, so the new network is not yet an exact reproduction. Recurrent states and persistence-event logic are otherwise left as in the original implementation.

In [ ]:
equations = """
dv/dt = (-v + ExtIn + P * (Inh + Exc)) / tau : 1
dExtIn/dt = -ExtIn / tau : 1
dInh/dt = -Inh / tau : 1
dExc/dt = -Exc / tau : 1
incoming_spikes : integer
P : integer
X : integer
Y : integer
"""
neurons = NeuronGroup(
    N, equations, threshold="v > vt",
    reset="v = vr; incoming_spikes_post = 0",
    refractory=0 * ms,
    events={"P_ON": "v > vt",
            "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
    method="exact",
)
neurons.X = np.arange(N) % width
neurons.Y = np.arange(N) // width
neurons.set_event_schedule("P_ON", when="after_thresholds")
neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
neurons.set_event_schedule("P_OFF", when="groups")
neurons.run_on_event("P_OFF", "P = 0", when="groups")

feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta")
feed.connect(j="i")
recurrent = Synapses(neurons, neurons, on_pre={
    "pre": "incoming_spikes_post += 1; Exc_pre = Wi",
    "pre_2": "Inh_post = P_post * Wk * incoming_spikes_post",
})
# All surrounding neighbors (no autapses), as in the original intended geometry.
for dy in range(-k, k + 1):
    for dx in range(-k, k + 1):
        if dx == dy == 0:
            continue
        xs = np.arange(max(0, -dx), min(width, width - dx))
        ys = np.arange(max(0, -dy), min(height, height - dy))
        if not len(xs) or not len(ys):
            continue
        xx, yy = np.meshgrid(xs, ys)
        src = (yy * width + xx).ravel()
        recurrent.connect(i=src, j=src + dy * width + dx)

## Simulate; verify output during camera silence

Continue the same network with no further camera spikes. Output during the silent interval is evidence of neuronal persistence, rather than an effect of video accumulation. This alone does not demonstrate velocity invariance. **At 640 x 480 and k=8, the full recurrent graph has tens of millions of edges and may require substantial memory and compute.**

In [ ]:
output = SpikeMonitor(neurons)
run(duration * second)
silence_duration = 0.1  # seconds
run(silence_duration * second)

output_t = np.asarray(output.t / second)
output_i = np.asarray(output.i, dtype=np.int64)
late = output_t >= duration + float(defaultclock.dt / second)
print(f"Input spikes: {len(pairs):,}")
print(f"Output spikes in silent interval: {np.count_nonzero(late):,}")
print(f"Neurons active during silence: {len(np.unique(output_i[late])):,}")

## Comparison video: camera events vs ESCM output

The screenshot showed activity confined to a narrow strip at the top of a mostly blank image. **That is a spatial framing issue, not a temporal window issue.** This cell reports the actual event-coordinate ranges and crops **both** panels to the same fixed bounding box of recorded input activity (with padding). The neural model, event coordinates, and 10 ms non-overlapping visualization windows remain unchanged. Set `crop_to_events = False` to inspect the entire 640 × 480 sensor frame. If the printed ranges are unexpectedly narrow, investigate the recording/decoder rather than treating cropping as a data correction.

In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

video_window_ms = 10.0
fps = 30
crop_to_events = True   # Display only: common fixed region for input and output
padding = 12            # Sensor pixels around observed input extent
video_path = root / "escm_input_vs_output.mp4"
window_s = video_window_ms / 1000
end_time = duration + silence_duration
input_t = pairs[:, 0].astype(float) * float(defaultclock.dt / second)
input_i = pairs[:, 1].astype(np.int64)
n_frames = int(np.ceil(end_time / window_s))

# Inspect the true coordinates before deciding how to frame the video.
xs = np.asarray(events["x"])[use]
ys = np.asarray(events["y"])[use]
print(f"Configured sensor: {width} x {height}")
print(f"Event extent: x={xs.min()}..{xs.max()}, y={ys.min()}..{ys.max()}")
if crop_to_events:
    x0, x1 = max(0, int(xs.min()) - padding), min(width, int(xs.max()) + padding + 1)
    y0, y1 = max(0, int(ys.min()) - padding), min(height, int(ys.max()) + padding + 1)
else:
    x0, x1, y0, y1 = 0, width, 0, height
view_w, view_h = x1 - x0, y1 - y0
print(f"Video display region: x={x0}:{x1}, y={y0}:{y1}")

with imageio.get_writer(video_path, fps=fps, codec="libx264", macro_block_size=16) as writer:
    for frame_id in range(n_frames):
        t0 = frame_id * window_s
        t1 = min((frame_id + 1) * window_s, end_time)
        a, b = np.searchsorted(input_t, (t0, t1), side="left")
        c, d = np.searchsorted(output_t, (t0, t1), side="left")

        input_frame = np.zeros(N, dtype=np.uint8)
        output_frame = np.zeros(N, dtype=np.uint8)
        input_frame[input_i[a:b]] = 255
        output_frame[output_i[c:d]] = 255
        left = input_frame.reshape(height, width)[y0:y1, x0:x1]
        right = output_frame.reshape(height, width)[y0:y1, x0:x1]

        frame = np.zeros((view_h + 48, 2 * view_w, 3), dtype=np.uint8)
        frame[48:, :view_w] = left[:, :, None]
        frame[48:, view_w:] = right[:, :, None]
        image = Image.fromarray(frame)
        draw = ImageDraw.Draw(image)
        draw.text((8, 6), f"Camera events ({video_window_ms:g} ms)", fill="white")
        draw.text((view_w + 8, 6), f"ESCM spikes ({video_window_ms:g} ms)", fill="white")
        draw.text((8, 26), f"t = {t0:.3f} - {t1:.3f} s", fill="white")
        if t0 >= duration:
            draw.text((view_w + 8, 26), "NO CAMERA INPUT", fill="white")
        writer.append_data(np.asarray(image))

print(f"Saved {n_frames} frames to {video_path}")